# 过度识别检验（计数数据）

> 对应代码：`MathStatsCode/code_in_notes/Chap.13/overid_nb_poi.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：分别用负二项数据（过度分散）与 Poisson 数据做 GMM 并执行过度识别检验，对比检验结论。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 用**过度识别检验**判断矩条件是否成立（作业：对比真正服从 Poisson 与过度分散的负二项数据）。
- `set obs 100`、`set seed 250211` 后，`gen x=rnbinomial(5,0.2)`：负二项分布。其期望为 np/(1-p)=5×0.8/0.2=20，但**方差大于均值**（过度分散）——Poisson 分布要求均值=方差，因此即使一阶矩相符，描述过度分散的第二个矩条件 `x²-(λ+λ²)` 会失效；
- `gmm (x-{lambda=1}) (x2-({lambda}+{lambda}²)), winit(identity) igmm` 用 Poisson 的两个理论矩做 GMM，`igmm` 表示迭代 GMM（不断更新权重矩阵）；`estat overid` 输出过度识别检验（Hansen's J）——由于数据不是 Poisson，检验应**拒绝**“矩条件成立”；
- 随后 `replace x=rpoisson(20)` 把数据换成真正的 Poisson(20)，重新估计并做 `estat overid`——此时检验应**不拒绝**。
两次结果的对比说明：过度识别检验能发现矩条件（模型设定）的错误，这是 GMM 相对于恰好识别方法的重要优势。


In [ ]:
%%stata
clear
set seed 250211
set obs 100
// 期望为5×0.8/0.2=20
gen x  = rnbinomial(5,0.2)
gen x2 = x^2
gmm (x-{lambda=1}) (x2-({lambda}+{lambda}^2)), winit(identity) igmm
estat overid 
replace x  = rpoisson(20)
replace x2 = x^2
gmm (x-{lambda=1}) (x2-({lambda}+{lambda}^2)), winit(identity) igmm
estat overid 
